In [1]:
import os
import json
import random
import re
from openai import OpenAI
import anthropic
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm
import time


# Load dataset

In [2]:
# Set Paths
base_dir = os.path.join(os.getcwd(), "dataset", "pororo")
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")

def load_dataset(qa_json_path, description_csv_path):
    try:
        # Load QA data
        with open(qa_json_path, "r") as f:
            qa_data = json.load(f)["PororoQA"]
        
        # Load descriptions
        descriptions = pd.read_csv(description_csv_path)
        
        return qa_data, descriptions
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], pd.DataFrame()
    
def get_random_questions(qa_data, max_questions=40, base_pattern="Pororo_ENGLISH1", seed=42):
    random.seed(seed)
    
    # Filter all eligible questions
    filtered_questions = [q for q in qa_data if base_pattern in q["video_name"]]
    
    # Group by unique (video_name, supporting_num) pairs to avoid duplicates
    unique_pairs = {}
    for q in filtered_questions:
        key = (q["video_name"], q["supporting_num"]) 
        if key not in unique_pairs:
            unique_pairs[key] = []
        unique_pairs[key].append(q)
    
    # Sample from unique pairs
    unique_keys = list(unique_pairs.keys())
    selected_keys = random.sample(unique_keys, min(max_questions, len(unique_keys)))
    
    # Get one question from each selected pair
    sampled_questions = []
    episode_counts = {}
    
    for key in selected_keys:
        question = random.choice(unique_pairs[key])
        sampled_questions.append(question)
        episode = question["video_name"]
        episode_counts[episode] = episode_counts.get(episode, 0) + 1
    
    # Group by season for display
    season_episodes = {
        "Pororo_ENGLISH1_1": [],
        "Pororo_ENGLISH1_2": [],
        "Pororo_ENGLISH1_3": []
    }
    
    for ep in episode_counts.keys():
        season = "_".join(ep.split("_")[:3])
        if season in season_episodes:
            season_episodes[season].append(ep)
    
    # Print statistics
    print(f"\nSelected {len(sampled_questions)} questions from {len(episode_counts)} episodes:")
    for season in sorted(season_episodes.keys()):
        season_eps = {ep: episode_counts[ep] for ep in season_episodes[season]}
        if season_eps:
            print(f"\n{season}:")
            for ep in sorted(season_eps.keys()):
                print(f"  {ep}: {season_eps[ep]} questions")
    
    return sampled_questions

def get_seeded_question(questions, gif_num, base_seed=42):
    if not questions:
        return None
    # Create new Random instance for each GIF
    local_random = random.Random(base_seed + gif_num)
    # Sort questions to ensure consistent ordering
    sorted_questions = sorted(questions, key=lambda x: x["qid"])
    return local_random.choice(sorted_questions)

# Load data
qa_data, descriptions = load_dataset(qa_json_path, description_csv_path)

# Get random sample of questions TODO increase number of questions
sampled_questions = get_random_questions(qa_data, max_questions=40)

# Group questions by supporting_num
grouped_questions = {}
for entry in sampled_questions:
    video_name = entry["video_name"]
    supporting_num = entry["supporting_num"]
    key = (video_name, supporting_num)
    if key not in grouped_questions:
        grouped_questions[key] = []
    grouped_questions[key].append(entry)

# Get unique pairs to process
gif_pairs = sorted(list(grouped_questions.keys()))
correct_count = 0
total_count = len(gif_pairs)

# Used to store question information for each GIF pair
question_data = {}  
for video_name, gif_num in gif_pairs:
    current_questions = grouped_questions[(video_name, gif_num)]
    if current_questions:
        entry = get_seeded_question(current_questions, int(gif_num))
        
        question = entry["question"]
        correct_idx = entry["correct_idx"]
        answers = [entry[f"answer{i}"] for i in range(5)]
        correct_answer = answers[correct_idx]
        qid = entry["qid"]

        question_data[(video_name, gif_num)] = {
            'entry': entry,
            'question': question,
            'correct_answer': correct_answer,
            'qid': qid
        }

evaluation_results = []




Selected 40 questions from 22 episodes:

Pororo_ENGLISH1_1:
  Pororo_ENGLISH1_1_ep1: 1 questions
  Pororo_ENGLISH1_1_ep10: 2 questions
  Pororo_ENGLISH1_1_ep11: 1 questions
  Pororo_ENGLISH1_1_ep12: 4 questions
  Pororo_ENGLISH1_1_ep13: 2 questions
  Pororo_ENGLISH1_1_ep2: 4 questions
  Pororo_ENGLISH1_1_ep5: 1 questions
  Pororo_ENGLISH1_1_ep6: 4 questions
  Pororo_ENGLISH1_1_ep9: 1 questions

Pororo_ENGLISH1_2:
  Pororo_ENGLISH1_2_ep10: 1 questions
  Pororo_ENGLISH1_2_ep2: 2 questions
  Pororo_ENGLISH1_2_ep5: 2 questions
  Pororo_ENGLISH1_2_ep8: 3 questions

Pororo_ENGLISH1_3:
  Pororo_ENGLISH1_3_ep1: 1 questions
  Pororo_ENGLISH1_3_ep11: 1 questions
  Pororo_ENGLISH1_3_ep12: 2 questions
  Pororo_ENGLISH1_3_ep13: 1 questions
  Pororo_ENGLISH1_3_ep2: 1 questions
  Pororo_ENGLISH1_3_ep3: 1 questions
  Pororo_ENGLISH1_3_ep4: 1 questions
  Pororo_ENGLISH1_3_ep5: 2 questions
  Pororo_ENGLISH1_3_ep7: 2 questions


# Single agent prediction

In [ ]:
load_dotenv()

# Configuration
# MODEL_NAME = "gpt-4o-mini"
MODEL_NAME = "claude-3-5-haiku-20241022"

# Determine which platform to use based on the model name
is_openai_model = not MODEL_NAME.startswith("claude-")

# Initialize appropriate client
if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

# Load descriptions
descriptions = pd.read_csv(description_csv_path)

def get_prediction(question, gif_paths, description, subtitles, max_retries=3, retry_delay=2):
    images = []
    for gif_path in gif_paths:
        with open(gif_path, "rb") as gif_file:
            images.append(gif_file.read())

    prompt = f"""
    As a cartoon analysis expert, answer the question strictly based on the visual content and available context using EXACTLY ONE SENTENCE:

    Input:
    Question: {question}
    Scene Description: {description}
    Subtitles: {subtitles}

    Guidelines:
    1. Consider cartoon-specific elements like character expressions, visual style, and narrative context.
    2. No explanations allowed.
    3. RESPONSE MUST BE IN ENGLISH ONLY. DO NOT INCLUDE TEXT IN ANY OTHER LANGUAGE.
    4. Avoid phrases like "Based on ...", "According to..." or "The description provided". 
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                # OpenAI implementation
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=150,
                    temperature=0.3,
                )
                response = completion.choices[0].message.content.strip().lower()
            else:
                # Anthropic implementation
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=150,
                    temperature=0.3,
                )
                response = completion.content[0].text.strip().lower()
            
            # Extract first sentence
            sentences = re.split(r'[.!?]', response)
            first_sentence = sentences[0].strip()
            
            # Skip empty sentences
            if not first_sentence and len(sentences) > 1:
                first_sentence = next((s.strip() for s in sentences if s.strip()), "")
                
            return first_sentence

        except Exception as e:
            print(f"Prediction attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    print(f"All {max_retries} attempts failed for question: {question}")
    return None

Using Anthropic model: claude-3-5-haiku-20241022


# Compute accuracy

In [ ]:
def compute_accuracy(question, correct_answer, predicted_answer, max_retries=2, retry_delay=2):

    prompt = f"""
    Evaluate the accuracy of the predicted answer according to criteria below:

    Input:
    Question: {question}
    Correct Answer: {correct_answer}
    Predicted Answer: {predicted_answer}

    Evaluation Rules:
    1. Be strict in your evaluation. The predicted answer must correctly address the question.
    2. Return ONLY a numeric score from [1.0, 0.75, 0.5, 0.25, 0.0].
    3. Do NOT include any explanation, words, punctuation, or additional content.

    Scoring Criteria:
    - 1.0: Contains the correct answer with the same core meaning as the reference
    - 0.75: Mostly correct with only minor differences that don't change the meaning
    - 0.5: Partially correct - contains some correct elements but misses important aspects
    - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
    - 0.0: Completely incorrect, contradicts the correct answer, or avoids answering by claiming insufficient information
    """
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=150,
                    temperature=0.3
                )
                response = completion.choices[0].message.content.strip()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=150,
                    temperature=0.3
                )
                response = completion.content[0].text.strip()

            numeric_match = re.search(r'(1\.0|0\.75|0\.5|0\.25|0\.0)', response)
            if numeric_match:
                score = float(numeric_match.group(1))
            else:
                score = 0.0

            return score

        except Exception as e:
            print(f"Accuracy calculation attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
    # Return 0 if it can't parse the score
    return 0.0


# Evaluate model performance

In [5]:
try:
    # Initialize counters
    correct_count = 0
    total_count = len(gif_pairs)
    # Process each video and GIF pair
    for video_name, gif_num in tqdm(gif_pairs, total=total_count):
        # Get question information
        if (video_name, gif_num) not in question_data:
            print(f"No question data found for {video_name} GIF {gif_num}")
            continue
            
        # Use retrieved question information
        q_info = question_data[(video_name, gif_num)]
        question = q_info['question']
        correct_answer = q_info['correct_answer']
        qid = q_info['qid']
        # Construct paths
        episode_parts = video_name.split("_")
        episode_folder = os.path.join(base_dir, "Scenes_Dialogues", 
                                "_".join(episode_parts[:-1]),
                                video_name)
        subtitles_path = os.path.join(episode_folder, "subtitles.txt")
        
        # Load subtitles
        with open(subtitles_path, "r") as f:
            subtitles = f.read()
        # Process current GIF
        gif_paths = [os.path.join(episode_folder, f"{gif_num}.gif")]

        # Get description
        description_rows = descriptions.loc[
            (descriptions.iloc[:, 0] == video_name) & 
            (descriptions.iloc[:, 1] == int(gif_num))
        ]
        if description_rows.empty:
            print(f"Description for {video_name} GIF {gif_num} not found")
            continue

        descriptions_list = description_rows.iloc[:, 2].tolist()
        description = " ".join(descriptions_list)

        # Get prediction
        predicted_answer = get_prediction(question, gif_paths, description, subtitles)
        # Calculate accuracy - ensure question parameter is passed
        is_correct = 0
        if predicted_answer is not None:
            is_correct = compute_accuracy(question, correct_answer, predicted_answer)
        correct_count += is_correct
        # Store current result
        result = {
            'gif_num': gif_num,
            'video_name': video_name,
            'qid': qid,
            'question': question,
            'correct_answer': correct_answer,
            'predicted_answer': predicted_answer,
            'accuracy': is_correct
        }
        evaluation_results.append(result)
        # Print debugging info
        print(f"\nVideo name: {video_name}")
        print(f"GIF number: {gif_num}")
        print(f"QID: {qid}")
        print(f"Question: {question}")
        print(f"Correct Answer: {correct_answer}")
        print(f"Predicted Answer: {predicted_answer}")
        print(f"Accuracy: {float(is_correct):.4f}")
    # Calculate overall accuracy
    average_accuracy = correct_count / total_count if total_count > 0 else 0
    print(f"\nAverage Accuracy: {average_accuracy:.4f}")

except Exception as e:
    print(f"Unexpected error in evaluation: {e}")
    average_accuracy = 0

  0%|          | 0/40 [00:00<?, ?it/s]

  2%|▎         | 1/40 [00:03<02:17,  3.53s/it]


Video name: Pororo_ENGLISH1_1_ep1
GIF number: 14
QID: 383
Question: whtat does eddy ask pororo
Correct Answer: he asks pororo what are you doing
Predicted Answer: based on the visual context, eddy appears to be asking crong about the reason behind pororo's urgent action
Accuracy: 0.2500


  5%|▌         | 2/40 [00:06<01:54,  3.00s/it]


Video name: Pororo_ENGLISH1_1_ep10
GIF number: 12
QID: 1100
Question: were eddy's friends interested seeing his new toy?
Correct Answer: yes, they ran happily towards the new toy.
Predicted Answer: yes, pororo and crong appear very interested and excited, eagerly running toward the car and engaging with eddy's new toy
Accuracy: 1.0000


...


Video name: Pororo_ENGLISH1_3_ep5
GIF number: 25
QID: 2333
Question: does the friends find pororo behind the snow man?
Correct Answer: the friends find pororo behind the snow man.
Predicted Answer: based on the scene description showing the friends waving goodbye to pororo, there is no visual indication that they find pororo behind a snowman
Accuracy: 0.0000


 98%|█████████▊| 39/40 [01:47<00:02,  2.63s/it]


Video name: Pororo_ENGLISH1_3_ep7
GIF number: 25
QID: 2425
Question: what does crong do when pororo says "come here"
Correct Answer: crong runs away from pororo
Predicted Answer: when pororo tells crong to come here, crong tries to run away and avoid following pororo's instruction
Accuracy: 1.0000


100%|██████████| 40/40 [01:49<00:00,  2.74s/it]


Video name: Pororo_ENGLISH1_3_ep7
GIF number: 47
QID: 2446
Question: what does poby say when invited to play
Correct Answer: he says "of course"
Predicted Answer: when invited to play, poby enthusiastically responds "of course" and joins his friends
Accuracy: 1.0000

Average Accuracy: 0.5437


# Save data

In [6]:
# Remove any existing Average rows
evaluation_results = [r for r in evaluation_results if r['gif_num'] != 'Average']

# Get unique videos
unique_videos = len(set(r['video_name'] for r in evaluation_results))

# Add row numbers to each result
for i, result in enumerate(evaluation_results, 1):
    result['row_num'] = i

# Add average accuracy as the last row
average_result = {
    'row_num': len(evaluation_results) + 1,
    'gif_num': 'Average',
    'video_name': f'Total Videos: {unique_videos}',
    'qid': '',
    'question': f'Total Questions: {len(evaluation_results)}',
    'correct_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy
}
evaluation_results.append(average_result)

# Define column order (reordered to put video_name before gif_num)
column_order = [
    'row_num',
    'video_name', 
    'gif_num',
    'qid',
    'question',
    'correct_answer',
    'predicted_answer',
    'accuracy'
]

# Create safe model name for file naming
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')

# Set up output directory
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
output_path = os.path.join(
    results_dir,
    f'pororo_single_agent_{safe_model_name}.csv'
)

# Remove existing file if it exists
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Save results with error handling
try:
    results_df = pd.DataFrame(evaluation_results)
    results_df = results_df[column_order]
    results_df.to_csv(output_path, index=False)
    
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/pororo_single_agent_claude_3_5_haiku_20241022.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/pororo_single_agent_claude_3_5_haiku_20241022.csv
